<a href="https://colab.research.google.com/github/iammuhammad41/Sentiment-Analysis-NLP-/blob/main/sentiment-analysis_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install nltk
# !pip install textblob
# !pip install wordcloud

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objs as go
import plotly.io as pio
import nltk
from nltk.corpus import stopwords
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import Word, TextBlob
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import cross_val_score, GridSearchCV, cross_validate
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from warnings import filterwarnings
filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [ ]:
df = pd.read_excel("../input/amazon-reviews/amazon.xlsx")
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(subset=['Review'], inplace=True)
df.isnull().sum()

In [ ]:
# to put all the characters in a standard
df['Review'] = df['Review'].str.lower()  #?make all lowercase

In [ ]:
# check sentences and replace punctions with spaces
df['Review'] = df['Review'].str.replace('[^\w\s]', '')

In [ ]:
df['Review'] = df['Review'].str.replace('\d', '')

In [ ]:
nltk.download('stopwords')

In [ ]:
#?save the downloaded stopwords
sw = stopwords.words('english')

In [ ]:
df['Review'] = df['Review'].apply(lambda x: " ".join(x for x in str(x).split() if x not in sw))

In [ ]:
#?to find how many times words counts
temp_df = pd.Series(' '.join(df['Review']).split()).value_counts()
temp_df

In [ ]:
# choose the words with less than 2 frequencies to drop
drops = temp_df[temp_df <= 1]

In [ ]:
df['Review'] = df['Review'].apply(lambda x: " ".join(x for x in x.split() if x not in drops))

In [ ]:
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
df['Review'] = df['Review'].apply(lambda x: " ".join([Word(word).lemmatize() for word in x.split()]))

In [ ]:
# extract the term frequencies(frequency of the words) and create a df
tf = df["Review"].apply(lambda x: pd.value_counts(x.split(" "))).sum(axis=0).reset_index()
tf.head()

In [ ]:
# fix the column names
tf.columns = ["words", "tf"]

In [ ]:
# to see the most frequent words
tf.sort_values("tf", ascending=False)

In [ ]:
tf[tf["tf"] > 300].plot.bar(x="words", y="tf")
plt.show()

In [ ]:
# check every row and join with these with a space
text = " ".join(i for i in df.Review)

In [ ]:
wordcloud = WordCloud().generate(text)
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

In [ ]:
# configure the graph
wordcloud = WordCloud(max_font_size=50,
                      max_words=100,
                      background_color="white").generate(text)
plt.figure()
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.show()

In [ ]:
df["Review"].head()

In [ ]:
nltk.download('vader_lexicon')  # pre-trained model for sentiment analysis

In [ ]:
#?Example 1
sia = SentimentIntensityAnalyzer()
sia.polarity_scores("The film was awesome")

In [ ]:
#?Example 2
sia.polarity_scores("I liked this music but it is not good as the other one")

In [ ]:
df["Review"][0:10].apply(lambda x: sia.polarity_scores(x))

In [ ]:
# take only compound scores
df["Review"][0:10].apply(lambda x: sia.polarity_scores(x)["compound"])

In [ ]:
#?save the scores as a new column
df["polarity_score"] = df["Review"].apply(lambda x: sia.polarity_scores(x)["compound"])

<a id = "5"></a>

<center><img src="https://learn.g2crowd.com/hubfs/unsupervised-learning.png" width="500"/></center>

In [ ]:
df["Review"][0:10].apply(lambda x: "pos" if sia.polarity_scores(x)["compound"] > 0 else "neg")

In [ ]:
rev_pol = pd.concat([df["Review"][0:10], df["Review"][0:10].apply(lambda x: "pos" if sia.polarity_scores(x)["compound"] > 0 else "neg")], axis=1)
rev_pol.columns = ["Review", "Polarity Scores"]
rev_pol

In [ ]:
df["sentiment_label"] = df["Review"].apply(lambda x: "pos" if sia.polarity_scores(x)["compound"] > 0 else "neg")
df["sentiment_label"].value_counts()

In [ ]:
df.groupby("sentiment_label")["Star"].mean()

In [ ]:
df["sentiment_label"] = LabelEncoder().fit_transform(df["sentiment_label"])

In [ ]:
y = df["sentiment_label"]
X = df["Review"]

In [ ]:
print(" X ".center(50, "~"))
display(X.head())
print("")
print(" Y ".center(50, "~"))
display(y.head())

In [ ]:
a = """ngram is a contiguous sequence of n items from a given sample of text or speech."""

TextBlob(a).ngrams(3)  # For example, let's create a triple ngram

In [ ]:
corpus = ['This is the first document.',
          'This document is the second document.',
          'And this is the third one.',
          'Is this the first document?']

In [ ]:
# word frequency
vectorizer = CountVectorizer()
X_c = vectorizer.fit_transform(corpus)  # we transformed the corpus with fit_transform
vectorizer.get_feature_names()

In [ ]:
X_c.toarray()  # we vectorized each unit

In [ ]:
# n-gram
vectorizer2 = CountVectorizer(analyzer='word', ngram_range=(2, 2))
X_n = vectorizer2.fit_transform(corpus)
vectorizer2.get_feature_names()  # it brought the words one by one above, now he brought them in phrases

In [ ]:
X_n.toarray()

In [ ]:
vectorizer = CountVectorizer()  # default => word count
X_count = vectorizer.fit_transform(X)  # X stands for texts, we only count the frequencies of the words

In [ ]:
vectorizer.get_feature_names()[10:15]  # let's look at some features

In [ ]:
X_count.toarray()[10:15]

In [ ]:
# word frequency
tf_idf_word_vectorizer = TfidfVectorizer()  #?default = word frequency
corpus_tf_idf_word = tf_idf_word_vectorizer.fit_transform(corpus)
tf_idf_word_vectorizer.get_feature_names_out()

In [ ]:
corpus_tf_idf_word.toarray()

In [ ]:
# ngram
tf_idf_ngram_vectorizer = TfidfVectorizer(ngram_range=(2, 3))
corpus_tf_idf_ngram = tf_idf_ngram_vectorizer.fit_transform(X)
tf_idf_ngram_vectorizer.get_feature_names_out()

In [ ]:
corpus_tf_idf_ngram.toarray()

In [ ]:
# word
tf_idf_word_vectorizer = TfidfVectorizer()
X_tf_idf_word = tf_idf_word_vectorizer.fit_transform(X)

In [ ]:
# ngram
tf_idf_ngram_vectorizer = TfidfVectorizer(ngram_range=(2, 3))
X_tf_idf_ngram = tf_idf_ngram_vectorizer.fit_transform(X)

In [ ]:
# Words with TF-IDF
log_model = LogisticRegression().fit(X_tf_idf_word, y)
cross_val_score(log_model,
                X_tf_idf_word,
                y,
                scoring="accuracy",
                cv=5).mean()

In [ ]:
# get a new comment
new_review = pd.Series("this product is great")

In [ ]:
# vectorize the new review
new_review = TfidfVectorizer().fit(X).transform(new_review)

# predict the sentiment of review
log_model.predict(new_review)

In [ ]:
# Let's pick a sample comment from the original dataset and ask it to the model
sample = df["Review"].sample(1).values
print(sample)
random_review = pd.Series(sample)
new_review = TfidfVectorizer().fit(X).transform(random_review)
log_model.predict(new_review)

In [ ]:
# Count Vectors
rf_model = RandomForestClassifier().fit(X_count, y)
print("Count Vectors Score", cross_val_score(rf_model, X_count, y, cv=5, n_jobs=-1).mean())

# TF-IDF Word-Level
rf_model = RandomForestClassifier().fit(X_tf_idf_word, y)
print("TF-IDF Word-Level Score", cross_val_score(rf_model, X_tf_idf_word, y, cv=5, n_jobs=-1).mean())

# TF-IDF N-Gram
rf_model = RandomForestClassifier().fit(X_tf_idf_ngram, y)
print("TF-IDF N-Gram Score", cross_val_score(rf_model, X_tf_idf_ngram, y, cv=5, n_jobs=-1).mean())

In [ ]:
%%time
def score(method_list):
    scores = []
    for method in method_list:
        rf_model = RandomForestClassifier().fit(method, y)
        scores.append(cross_val_score(rf_model, method, y, cv=5, n_jobs=-1).mean())

    scores_df = pd.DataFrame({"Methods": ["Count Vectors", "TF-IDF Word-Level", "TF-IDF N-Gram"],
                             "Scores": scores})

    trace1 = go.Bar(
                    y = scores_df["Methods"],
                    x = scores_df["Scores"],
                    name = "Accuracy Plot",
                    text=[round(i,5) for i in scores_df['Scores']],
                    marker = dict(color = ['#2ECC71','#34495E','#D0D3D4'],
                                 line=dict(color='rgb(0,0,0)',width=1.5)),
        orientation='h', textposition = 'inside'
    )
    data = [trace1]
    layout = go.Layout(barmode = "group",
                       title={'text': "Scores" ,
                                 'y':0.9,
                                 'x':0.5,
                                 'xanchor': 'center',
                                 'yanchor': 'top'},
                          template='plotly_white')
    fig = go.Figure(data = data, layout = layout)
    pio.show(fig)

method_list = [X_count, X_tf_idf_word, X_tf_idf_ngram]
score(method_list)

<div class="alert alert-block alert-info">
        <b> ?? The 3 methods above were the feature generation method. We built models for each and looked at 5-fold cross validation.</b>
</div>
<br>

<span class="label label-default"
     style="color:white;
           display:fill;
           border-radius:5px;
           background-color:crimson;
           font-size:120%;
           font-family:Verdana;
           letter-spacing:0.5px">
Hyperparameter Optimization
</span>

Since we got the highest success from the Count Vectors model, let's continue with it. Firstly, let's explain the method that we will use called as **GridSearchCV**.

<span style="color:#FF0000"><b>GridSearchCV:</b></span> A separate model is created with all combinations for the hyperparameters and their values that are desired to be tested in the model, and the most successful hyperparameter set is determined according to the specified metric. Simply, we will give a parameter set, and the method will try the all of the values we give on the set, and get the best result.

In [ ]:
rf_model = RandomForestClassifier(random_state=17)  # create an empty model object

We apply GridSearchCV with a small number of parameter values in order not to prolong the process. More parameters and more values can be experimented with, this will increase the processing time so we give a brief overview here.

In [ ]:
%%time
rf_params = {"max_depth": [8, None],
             "max_features": [7, "auto"],
             "min_samples_split": [2, 5, 8],
             "n_estimators": [100, 200]
             }

rf_best_grid = GridSearchCV(rf_model,
                            rf_params,
                            cv=5,
                            n_jobs=-1,
                            verbose=1).fit(X_count, y)

In [ ]:
# best parameters that GridSearchCV found
rf_best_grid.best_params_

In [ ]:
#?set the model with the parameters GridSearchCV found above
rf_final = rf_model.set_params(**rf_best_grid.best_params_, random_state=17).fit(X_count, y)

# our final score
cross_val_score(rf_final, X_count, y, cv=5, n_jobs=-1).mean()

<center style="font-family:cursive; font-size:18px; color:#159364;">Please vote and comment if you enjoyed my kernel, thanks for reading! ??</center>